#<font color="blue"><b>******* Instruction Fine-Tuning *******</b></font>

### <font color="blue"><b> Import Required Libraries </b></font>

In [ ]:
import random
import torch
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

### <font color="blue"><b> Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET = "databricks/databricks-dolly-15k"
MAX_LEN = 256
IGNORE_INDEX = -100

### <font color="blue"><b> Load the Pretrained Tokenizer and Model

In [ ]:
# ---------------------------------------------------------------------------
# Load model and tokenizer
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

### <font color="blue"><b> Encode Instruction and Context – Response(output) Pairs

In [ ]:
# ---------------------------------------------------------------------------
# Build one user turn from instruction + optional context
# ---------------------------------------------------------------------------
def build_user_turn(instruction, context):
    if context:
        return instruction + "\n\n" + context
    return instruction


# ---------------------------------------------------------------------------
# Turn one (instruction, context, output) example into token ids + labels
# ---------------------------------------------------------------------------
def encode_example(instruction, context, assistant_msg):
    user_msg = build_user_turn(instruction, context)

    messages_full = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    # We still need prompt_text, but only to get its CHARACTER length —
    # not to derive a token count from a second, independent tokenization.
    messages_prompt_only = [{"role": "user", "content": user_msg}]
    prompt_text = tokenizer.apply_chat_template(messages_prompt_only, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages_full, tokenize=False, add_generation_prompt=False)

    # Tokenize full_text ONCE, with character offsets per token.
    encoding = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
    full_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    # Truncate to max length first
    input_ids = full_ids[:MAX_LEN]
    offsets = offsets[:MAX_LEN]

    # Find the token index where the response actually starts, using the
    # prompt's character length as the boundary — robust to BPE merges
    # across the prompt/response seam.
    prompt_char_len = len(prompt_text)
    prompt_len = len(input_ids)
    for i, (start, end) in enumerate(offsets):
        if start >= prompt_char_len:
            prompt_len = i
            break

    labels = []
    for i in range(len(input_ids)):
        if i < prompt_len:
            labels.append(IGNORE_INDEX)
        else:
            labels.append(input_ids[i])

    # Track real content length BEFORE padding — this is what makes the
    # attention mask correct even when pad_token_id == eos_token_id.
    actual_len = len(input_ids)

    while len(input_ids) < MAX_LEN:
        input_ids.append(tokenizer.pad_token_id)
        labels.append(IGNORE_INDEX)

    # Build attention mask from the real content length, NOT by comparing
    # token values to pad_token_id (which is ambiguous when pad == eos).
    attention_mask = [1] * actual_len + [0] * (MAX_LEN - actual_len)

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

### <font color="blue"><b> Create a Custom Dataset

In [ ]:
# ---------------------------------------------------------------------------
# Simple dataset wrapper
# ---------------------------------------------------------------------------
class IFTDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        return encode_example(example["instruction"], example["context"], example["output"])

### <font color="blue"><b> Load the Dolly-15K Dataset

In [ ]:
# ---------------------------------------------------------------------------
# Load the Dolly-15k dataset
# ---------------------------------------------------------------------------
raw_data = load_dataset(DATASET, split="train")

all_examples = []
for row in raw_data:
    example = {
        "instruction": row["instruction"],
        "context": row["context"],
        "output": row["response"],
    }
    all_examples.append(example)

random.seed(0)
random.shuffle(all_examples)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

### <font color="blue"><b> Shuffle and Split the Dataset


In [ ]:
# ---------------------------------------------------------------------------
# Split into train / validation / test
# ---------------------------------------------------------------------------
N_TRAIN = 200
N_VAL = 20
N_TEST = 5

train_examples = all_examples[:N_TRAIN]
val_examples = all_examples[N_TRAIN:N_TRAIN + N_VAL]
test_examples = all_examples[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Test examples:", len(test_examples))

train_dataset = IFTDataset(train_examples)
val_dataset = IFTDataset(val_examples)

Training examples: 200
Validation examples: 20
Test examples: 5


### <font color="blue"><b> Configure Training </b></font>
### <font color="blue"><b> Initialize the Trainer</b></font>
### <font color="blue"><b> Fine-Tune the Model</b></font>

In [ ]:
# ---------------------------------------------------------------------------
# Train (with validation each epoch)
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./ift_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
trainer.train()
trainer.save_model("./ift_out")

final_val_metrics = trainer.evaluate()
print("\nFinal validation metrics:", final_val_metrics)

Epoch,Training Loss,Validation Loss
1,1.957084,2.458712
2,1.743808,2.385866
3,1.458681,2.385436


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
1.458681,2.385436,3



Final validation metrics: {'eval_loss': 2.3854355812072754}


### <font color="blue"><b> Generate Responses </b></font>

1. Formats the instruction using the Qwen chat template.
2. Tokenizes the prompt.
3. Generates new tokens using autoregressive decoding.
4. Converts the generated tokens back into readable text.

In [ ]:
# ---------------------------------------------------------------------------
# Simple qualitative evaluation on the held-out test set
# ---------------------------------------------------------------------------
def generate_response(instruction, context):
    user_msg = build_user_turn(instruction, context)
    messages = [{"role": "user", "content": user_msg}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### <font color="blue"><b> Evaluate on the Test Set

Finally, the fine-tuned model is evaluated using previously unseen test examples.

For each test sample, the notebook displays:

- The user instruction
- The reference response from the dataset
- The model-generated response

This qualitative evaluation helps assess how well the model has learned the instruction-following task.

In [ ]:
print("\n=== Test Set Evaluation ===")
for example in test_examples:
    prediction = generate_response(example["instruction"], example["context"])
    print("\nInstruction:", example["instruction"])
    print("Context    :", example["context"])
    print("Reference  :", example["output"])
    print("Prediction :", prediction)


=== Test Set Evaluation ===

Instruction: What are the Olympic Games?
Context    : 
Reference  : The modern Olympic Games or Olympics (French: Jeux olympiques)are the leading international sporting events featuring summer and winter sports competitions in which thousands of athletes from around the world participate in a variety of competitions. The Olympic Games are considered the world's foremost sports competition with more than 200 teams, representing sovereign states and territories, participating. The Olympic Games are normally held every four years, and since 1994, have alternated between the Summer and Winter Olympics every two years during the four-year period.
Prediction : The Olympic Games are an international sporting event that is held every four years. The first Olympic Games were held in 1896 in Athens, Greece. The Olympic Games are held to celebrate the achievements of humanity and to promote friendship, understanding and peace. The Olympic Games are held in the summer